# Notebook 03: Spatial Mapping and Geostatistics

This notebook demonstrates the **geostatistical mapping pipeline** from the
`ragweed_toolkit.geostatistics` and `ragweed_toolkit.spatial` modules
(Chapter 5, Act I -- Field Detection).

The workflow transforms point-based weed detections into:

1. **Semivariogram analysis** -- quantify spatial autocorrelation
2. **Ordinary kriging** -- interpolate a continuous density surface
3. **Bivariate LISA** -- identify hot spots linking satellite and weed data
4. **GWR** -- model spatially varying relationships (satellite vs weed density)

All data is synthetic -- no field data or external files are needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, box

np.random.seed(42)

## 1. Generate Synthetic Spatial Data

Simulate 150 observation points across a 400m x 400m field with spatially
correlated ragweed density. A hot spot is placed near the center.

In [ ]:
n_points = 150
field_size = 400  # metres

# Random observation locations within the field
x = np.random.uniform(0, field_size, n_points)
y = np.random.uniform(0, field_size, n_points)
coords = np.column_stack([x, y])

# Create a hot spot centered at (250, 200)
hotspot_x, hotspot_y = 250, 200
dist_to_hotspot = np.sqrt((x - hotspot_x)**2 + (y - hotspot_y)**2)

# Weed density: exponential decay from hotspot + random noise
density = np.maximum(0, 80 * np.exp(-dist_to_hotspot / 100) + np.random.randn(n_points) * 12)

print(f"=== Synthetic Field Data ===")
print(f"  Points    : {n_points}")
print(f"  Field     : {field_size}m x {field_size}m")
print(f"  Density   : min={density.min():.1f}, mean={density.mean():.1f}, max={density.max():.1f}")
print(f"  Hotspot   : ({hotspot_x}, {hotspot_y})")

## 2. Semivariogram Analysis

The experimental semivariogram quantifies how spatial dependence changes with
distance. We then fit an exponential model:

$$\gamma(h) = C_0 + C \cdot [1 - \exp(-h / a)]$$

where $C_0$ = nugget, $C$ = partial sill, $a$ = range.

In [ ]:
from ragweed_toolkit.geostatistics import (
    compute_experimental_variogram,
    fit_exponential_model,
    plot_variogram,
)

# Compute experimental variogram
lag_centers, gamma, pair_counts = compute_experimental_variogram(
    coords, density, n_lags=12, max_lag=200.0
)

print("=== Experimental Variogram ===")
print(f"  {'Lag (m)':>10} {'Semivariance':>14} {'Pairs':>8}")
print(f"  {'-'*10} {'-'*14} {'-'*8}")
for lag, g, c in zip(lag_centers, gamma, pair_counts):
    g_str = f"{g:.1f}" if not np.isnan(g) else "NaN"
    print(f"  {lag:>10.1f} {g_str:>14} {c:>8d}")

In [ ]:
# Fit exponential model
model = fit_exponential_model(lag_centers, gamma)

print("=== Fitted Exponential Model ===")
print(f"  Nugget (C0)     : {model.nugget:.1f}")
print(f"  Partial sill (C): {model.partial_sill:.1f}")
print(f"  Sill (C0+C)     : {model.sill:.1f}")
print(f"  Range (a)       : {model.range_param:.1f} m")

# Plot
from ragweed_toolkit.viz.style import set_publication_style
set_publication_style()

fig, ax = plt.subplots(figsize=(9, 5))
plot_variogram(model, title="Ragweed Density Semivariogram", ax=ax)
plt.tight_layout()
plt.show()

## 3. Ordinary Kriging

Interpolate the point observations onto a regular 10m grid clipped to
the field boundary. The result is a continuous density surface with
prediction variance at each cell.

In [ ]:
from ragweed_toolkit.geostatistics import ordinary_kriging

# Build GeoDataFrame from observations
gdf_obs = gpd.GeoDataFrame(
    {"density": density},
    geometry=[Point(xi, yi) for xi, yi in zip(x, y)],
    crs="EPSG:32719",
)

# Field boundary polygon
field_boundary = gpd.GeoDataFrame(
    geometry=[box(0, 0, field_size, field_size)],
    crs="EPSG:32719",
)

# Run kriging with the fitted variogram
grid = ordinary_kriging(
    gdf_obs,
    value_col="density",
    boundary=field_boundary,
    resolution=10.0,
    variogram_model=model,
)

print(f"=== Kriging Result ===")
print(f"  Grid cells      : {len(grid)}")
print(f"  Predicted range : [{grid['predicted'].min():.1f}, {grid['predicted'].max():.1f}]")
print(f"  Variance range  : [{grid['variance'].min():.1f}, {grid['variance'].max():.1f}]")

In [ ]:
# Plot kriged density surface
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Predicted density
grid.plot(
    column="predicted", cmap="YlOrRd", legend=True, ax=axes[0],
    legend_kwds={"label": "Predicted density", "shrink": 0.7},
)
axes[0].scatter(x, y, c="black", s=5, alpha=0.5, label="Observations")
axes[0].set_title("(a) Kriged Density Surface", fontweight="bold")
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")
axes[0].legend(loc="upper left")

# Panel B: Prediction variance
grid.plot(
    column="variance", cmap="Blues", legend=True, ax=axes[1],
    legend_kwds={"label": "Variance", "shrink": 0.7},
)
axes[1].scatter(x, y, c="black", s=5, alpha=0.5)
axes[1].set_title("(b) Prediction Variance", fontweight="bold")
axes[1].set_xlabel("Easting (m)")
axes[1].set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

## 4. Bivariate LISA Cluster Analysis

LISA (Local Indicators of Spatial Association) identifies statistically
significant spatial clusters where two variables co-occur:

| Cluster | Meaning | Action |
|---------|---------|--------|
| HH (red) | Hot spot -- high satellite index, high weed density | Priority treatment |
| LL (blue) | Cold spot -- low satellite, low density | Monitor only |
| HL (orange) | High satellite, low density neighbors | Investigate |
| LH (purple) | Low satellite, high density neighbors | Emerging outbreak |

In [ ]:
# Generate satellite-derived variable correlated with weed density
# Simulates PRESTO PC1 values
pc1 = 0.7 * density / density.max() + 0.3 * np.random.randn(n_points) * 0.2

gdf_spatial = gpd.GeoDataFrame(
    {"pc1": pc1, "density": density},
    geometry=[Point(xi, yi) for xi, yi in zip(x, y)],
    crs="EPSG:32719",
)

print(f"=== Bivariate Data ===")
print(f"  PC1 range     : [{pc1.min():.3f}, {pc1.max():.3f}]")
print(f"  Density range : [{density.min():.1f}, {density.max():.1f}]")
print(f"  Correlation   : {np.corrcoef(pc1, density)[0,1]:.3f}")

In [ ]:
from ragweed_toolkit.spatial import compute_bivariate_morans, compute_bivariate_lisa, lisa_summary
from ragweed_toolkit.spatial.lisa import LISA_COLORS

# Global bivariate Moran's I
bv_result = compute_bivariate_morans(
    gdf_spatial, var_x="pc1", var_y="density",
    threshold=80.0, permutations=999,
)

print("=== Bivariate Moran's I (PC1 x Density) ===")
print(f"  I          = {bv_result.I:.3f}")
print(f"  z-score    = {bv_result.z_score:.3f}")
print(f"  p-value    = {bv_result.p_value:.4f}")
print(f"  Significant: {bv_result.significant} ({bv_result.significance_stars})")

In [ ]:
# LISA decomposition
lisa_gdf = compute_bivariate_lisa(
    gdf_spatial, var_x="pc1", var_y="density",
    threshold=80.0, permutations=999, alpha=0.05,
)

summary = lisa_summary(lisa_gdf)
print("=== LISA Clusters ===")
for cluster, stats in summary.items():
    color = LISA_COLORS[cluster]
    print(f"  {cluster:3s} : {stats['count']:4d} points ({stats['pct']:5.1f}%)  {color}")

In [ ]:
# Plot LISA clusters
from ragweed_toolkit.viz.style import LISA_COLORS as VIZ_LISA_COLORS, LISA_LABELS

fig, ax = plt.subplots(figsize=(9, 8))

for cluster in ["HH", "LL", "HL", "LH", "NS"]:
    mask = lisa_gdf["cluster"] == cluster
    if mask.sum() == 0:
        continue
    subset = lisa_gdf[mask]
    ax.scatter(
        [p.x for p in subset.geometry],
        [p.y for p in subset.geometry],
        c=VIZ_LISA_COLORS[cluster],
        s=40 if cluster != "NS" else 15,
        alpha=0.9 if cluster != "NS" else 0.3,
        label=f"{LISA_LABELS[cluster]} (n={mask.sum()})",
        edgecolors="black", linewidths=0.3,
    )

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title("Bivariate LISA Clusters (PC1 x Weed Density)", fontweight="bold")
ax.legend(title="Cluster", loc="upper left", fontsize=8)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 5. GWR vs OLS Comparison

Geographically Weighted Regression (GWR) allows regression coefficients to
vary spatially, capturing local relationships that OLS misses.

**Note**: This section requires `mgwr` and `spreg`. If not installed, it will
be skipped gracefully.

In [ ]:
# Add a second predictor (e.g., soil moisture index)
moisture = np.sin(x / 120) * np.cos(y / 150) + np.random.randn(n_points) * 0.15
gdf_spatial["moisture"] = moisture

try:
    from ragweed_toolkit.spatial import fit_ols, fit_gwr, compare_ols_gwr

    comparison = compare_ols_gwr(
        gdf_spatial,
        y_col="density",
        x_cols=["pc1", "moisture"],
    )

    print("=== OLS vs GWR Comparison ===")
    print(f"\n  OLS:")
    print(f"    R-squared     : {comparison.ols.r2:.3f}")
    print(f"    Adj R-squared : {comparison.ols.adj_r2:.3f}")
    print(f"    AIC           : {comparison.ols.aic:.1f}")
    print(f"    Coefficients  : {comparison.ols.coefficients}")

    print(f"\n  GWR:")
    print(f"    R-squared     : {comparison.gwr.r2:.3f}")
    print(f"    Adj R-squared : {comparison.gwr.adj_r2:.3f}")
    print(f"    AICc          : {comparison.gwr.aicc:.1f}")
    print(f"    Bandwidth     : {comparison.gwr.bandwidth:.0f} neighbors")

    print(f"\n  Improvement:")
    print(f"    R2 gain       : +{comparison.r2_improvement:.3f}")
    print(f"    AICc delta    : {comparison.aicc_delta:.1f}")

except ImportError:
    print("mgwr/spreg not installed. Install with:")
    print('  pip install ragweed-ai-toolkit[spatial]')
    print("\nSkipping GWR comparison -- see the chapter for results.")

In [ ]:
# Plot local R-squared from GWR (if available)
try:
    local_r2 = comparison.gwr.local_r2

    fig, ax = plt.subplots(figsize=(9, 8))
    sc = ax.scatter(
        x, y, c=local_r2, cmap="RdYlGn", s=50,
        edgecolors="black", linewidths=0.3, vmin=0, vmax=1,
    )
    plt.colorbar(sc, ax=ax, label="Local R-squared", shrink=0.8)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_title("GWR Local R-squared", fontweight="bold")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

except NameError:
    print("GWR results not available (mgwr not installed).")

## Summary

This notebook demonstrated the spatial analysis pipeline:

1. **Semivariogram fitting** quantifies spatial dependence with nugget, sill, and range parameters.

2. **Ordinary kriging** interpolates point observations into a continuous density surface
   with prediction variance at each cell.

3. **Bivariate LISA** identifies statistically significant clusters where satellite indices
   and weed density co-occur (hot spots, cold spots, and spatial outliers).

4. **GWR** captures spatially varying relationships that OLS misses, typically improving
   R-squared by 0.2--0.3 (see Chapter 5, Table 8).

### Management Zones

The LISA clusters directly inform management:
- **HH zones** (red): Priority treatment areas
- **LL zones** (blue): Low-risk, monitoring only
- **HL/LH outliers**: Investigate for emerging outbreaks or resistant patches

### Next Steps

- **Notebook 04**: Connect satellite spectral indices to the spatial analysis pipeline